# FAN landmarks of the clean faces (as the original SMIRK preprocessing)

SMIRK's landmark loss uses the 17 jaw-line points of FAN (MediaPipe has no jaw line) and 5 FAN points to align the
MICA input. Same tools as `datasets/preprocess_scripts/apply_fan_to_dataset.py`: ibug RetinaFace (mobilenet0.25) +
ibug FAN (2dfan2_alt), https://github.com/hhj1897.

Inputs: `clean-faces`, `clean-faces-rafdb`, `expw-crops`, `affectnet_1`, `rafdb-crops` (all datasets in one run).
Settings: **GPU T4 x1**, **Internet on**. Output `/kaggle/working/fan_landmarks/<dataset>_fan.npz` → dataset `fan-landmarks`.

## 1. Code and ibug face detection / alignment

In [ ]:
import os, sys, glob
os.chdir('/kaggle/working')
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1

# the detector weights are stored with Git LFS
!apt-get -qq install -y git-lfs > /dev/null && git lfs install > /dev/null
!rm -rf /tmp/face_detection /tmp/face_alignment
!git clone -q https://github.com/hhj1897/face_detection.git /tmp/face_detection && cd /tmp/face_detection && git lfs pull
!git clone -q https://github.com/hhj1897/face_alignment.git /tmp/face_alignment && cd /tmp/face_alignment && git lfs pull
!pip install -q --no-deps -e /tmp/face_detection -e /tmp/face_alignment
!find /tmp/face_detection /tmp/face_alignment -name "*.pth" -size +1M | head
# the running kernel does not see packages installed after it started: check (and run the scripts) in new processes
os.environ['PYTHONPATH'] = '/tmp/face_detection:/tmp/face_alignment:' + os.environ.get('PYTHONPATH', '')
!python -c "import ibug.face_detection, ibug.face_alignment; print('ibug packages OK')"

## 2. Inputs

In [ ]:
def dataset_of(path):
    path = os.path.relpath(path, '/kaggle/input').lower()
    return next((k for k in ('expw', 'affectnet', 'rafdb') if k in path), None)

CLEAN_DIRS = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True))
IMAGE_DIRS = {dataset_of(p): p for p in glob.glob('/kaggle/input/**/images', recursive=True) if os.path.isdir(p) and dataset_of(p)}
print('clean dirs:', CLEAN_DIRS); print('image dirs:', IMAGE_DIRS)
assert CLEAN_DIRS and IMAGE_DIRS
CLEAN_ARGS = ' '.join(f'--clean_dir "{d}"' for d in CLEAN_DIRS)
OUT = '/kaggle/working/fan_landmarks'

## 3. Quick check (50 faces of each dataset) and a look at the landmarks

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
for name, folder in IMAGE_DIRS.items():
    !python /tmp/smirk/datasets/preprocess_scripts/extract_fan_landmarks.py {CLEAN_ARGS} --image_dir "{folder}" --dataset {name} --out /tmp/fan_check/{name}_fan.npz --limit 50

fig, axes = plt.subplots(len(IMAGE_DIRS), 6, figsize=(18, 3.2 * len(IMAGE_DIRS)), squeeze=False)
for row, (name, folder) in zip(axes, IMAGE_DIRS.items()):
    data = np.load(f'/tmp/fan_check/{name}_fan.npz')
    for ax, n, lmk, ok in zip(row, data['names'], data['landmarks'], data['detected']):
        ax.imshow(cv2.imread(f'{folder}/{n}.png')[..., ::-1]); ax.axis('off')
        if ok:
            ax.scatter(lmk[:17, 0], lmk[:17, 1], s=6, c='#eb6834'); ax.scatter(lmk[17:, 0], lmk[17:, 1], s=4, c='#2a78d6')
        ax.set_title(f'{name}: {n}', fontsize=8)
plt.tight_layout(); plt.show()

## 4. All clean faces (~30-60 min)

Long cells can stop in an interactive session: run this part with *Save Version → Save & Run All*.
The results are written every 2000 faces; a stopped run continues where it stopped when the cell is run again
(in the same session, or with the previous output added as input and copied to `OUT`).

In [ ]:
# continue from a previous (interrupted) run given as input
os.makedirs(OUT, exist_ok=True)
for previous in glob.glob('/kaggle/input/**/fan_landmarks/*_fan.npz', recursive=True):
    if not os.path.exists(f'{OUT}/{os.path.basename(previous)}'):
        !cp "{previous}" {OUT}/
for name, folder in IMAGE_DIRS.items():
    !python /tmp/smirk/datasets/preprocess_scripts/extract_fan_landmarks.py {CLEAN_ARGS} --image_dir "{folder}" --dataset {name} --out {OUT}/{name}_fan.npz
!ls -la {OUT}

## 5. Save

*Save Version → Save & Run All* (GPU T4 x1), then on the finished version *Output → New Dataset* `fan-landmarks` (Private).